# Predicting MLB World Series Winners from Regular-Season Performance

**Research question:** How accurately can an MLB playoff team's regular-season win percentage and performance statistics predict whether it will win the World Series, using data from the 2005–2024 MLB seasons?

**Author:** Nick Hawkins  
**Course:** DTSC 2301 — Modeling and Society

This notebook is written as a reproducible walkthrough. It prepares the data, explores the class imbalance, builds a baseline, compares logistic regression and random forest models, and interprets the results.

## 1. Problem definition

This is a **binary-classification** problem. Each row is one MLB playoff team in one season. The target, `world_series_winner`, equals 1 for the champion and 0 for every other playoff team.

The intended users are analysts or fans interested in how much regular-season strength carries into October. The model is descriptive and educational; it should not be treated as gambling advice.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    balanced_accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA = ROOT / "data"
FIGURES = ROOT / "assets" / "images"
FEATURES = ["win_pct", "run_diff_per_game", "runs_per_game", "ops",
            "home_runs_per_game", "stolen_bases_per_game", "era", "whip"]
sns.set_theme(style="whitegrid")

## 2. Data preparation

The raw file contains all 30 MLB teams for each season. I remove duplicate team-season rows, identify playoff participants using the postseason indicator fields, and calculate rate statistics so the shortened 2020 season remains comparable.

I do **not** use division rank, playoff round reached, league-champion status, or World Series results as features. Those fields would leak postseason information into the model.

In [ ]:
teams = pd.read_csv(DATA / "teams_2005_2024.csv")
teams = teams.drop_duplicates(subset=["yearID", "teamID"], keep="last").copy()

playoff_flag = teams[["DivWin", "WCWin", "LgWin", "WSWin"]].eq("Y").any(axis=1)
df = teams.loc[playoff_flag].copy()
df["world_series_winner"] = df["WSWin"].eq("Y").astype(int)

df["win_pct"] = df["W"] / df["G"]
df["run_diff_per_game"] = (df["R"] - df["RA"]) / df["G"]
df["runs_per_game"] = df["R"] / df["G"]
singles = df["H"] - df["2B"] - df["3B"] - df["HR"]
df["obp"] = (df["H"] + df["BB"] + df["HBP"]) / (df["AB"] + df["BB"] + df["HBP"] + df["SF"])
df["slg"] = (singles + 2*df["2B"] + 3*df["3B"] + 4*df["HR"]) / df["AB"]
df["ops"] = df["obp"] + df["slg"]
df["home_runs_per_game"] = df["HR"] / df["G"]
df["stolen_bases_per_game"] = df["SB"] / df["G"]
df["era"] = pd.to_numeric(df["ERA"], errors="coerce")
df["whip"] = (df["BBA"] + df["HA"]) / (df["IPouts"] / 3)

keep = ["yearID", "teamID", "name", "W", "L", "world_series_winner"] + FEATURES
df = df[keep].sort_values(["yearID", "win_pct"], ascending=[True, False]).reset_index(drop=True)
print(df.shape)
print(df["world_series_winner"].value_counts())

(198, 14)\nworld_series_winner\n0    178\n1     20\nName: count, dtype: int64\n

## 3. Data understanding

Only 20 of 198 playoff team-seasons are champions, so the classes are highly imbalanced. Ordinary accuracy can be misleading: a model that predicts “non-winner” for every row would be more than 89% accurate while identifying no champions.

World Series winners averaged a .597 regular-season winning percentage, compared with .576 for other playoff teams. The overlap is still substantial, so win percentage alone cannot guarantee a championship.

In [ ]:
summary = df.groupby("world_series_winner")[FEATURES].agg(["mean", "median"])
summary.loc[:, ["win_pct", "run_diff_per_game", "ops", "era", "whip"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="world_series_winner", y="win_pct", ax=axes[0], color="#13274F")
axes[0].set_xticklabels(["Other playoff teams", "Champions"])
axes[0].set_title("Win percentage by postseason outcome")
sns.scatterplot(data=df, x="win_pct", y="run_diff_per_game",
                hue="world_series_winner", palette={0:"#13274F", 1:"#CE1141"}, ax=axes[1])
axes[1].set_title("Regular-season strength overlaps")
plt.tight_layout();

## 4. Training and testing strategy

I use a chronological split instead of a random split. Seasons 2005–2019 train the models; seasons 2020–2024 form an untouched test set. This prevents information from later seasons from influencing earlier predictions and keeps every team from the same season in one split.

Missing numeric values, if any, are filled using the training-set median. Logistic-regression inputs are standardized inside the pipeline. The target never enters preprocessing.

In [ ]:
train = df[df["yearID"] <= 2019].copy()
test = df[df["yearID"] >= 2020].copy()
X_train, y_train = train[FEATURES], train["world_series_winner"]
X_test, y_test = test[FEATURES], test["world_series_winner"]
print("Training rows:", len(train), "Test rows:", len(test))
print("Training seasons:", train.yearID.min(), "to", train.yearID.max())
print("Test seasons:", test.yearID.min(), "to", test.yearID.max())

Training rows: 136 Test rows: 62\nTraining seasons: 2005 to 2019\nTest seasons: 2020 to 2024\n

## 5. Baseline and model development

The baseline selects the playoff team with the best regular-season winning percentage in each season. This is more informative than always predicting the majority class.

I compare two models:

1. **Logistic regression**, an interpretable linear probability model.
2. **Random forest**, a nonlinear model that can capture interactions.

Because exactly one team wins each season, every method ranks the playoff teams and selects its highest-scoring team as that season's predicted champion.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
prep = ColumnTransformer([("numbers", numeric_pipe, FEATURES)])

logit = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))
])
forest = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                      class_weight="balanced", random_state=42))
])
logit.fit(X_train, y_train)
forest.fit(X_train, y_train)

In [ ]:
def pick_one_per_season(frame, scores):
    ranked = frame[["yearID"]].copy()
    ranked["score"] = np.asarray(scores)
    chosen = ranked.groupby("yearID")["score"].idxmax()
    pred = pd.Series(0, index=frame.index, dtype=int)
    pred.loc[chosen] = 1
    return pred.to_numpy()

def score_model(name, y, pred, prob):
    return {"model": name,
            "accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, prob),
            "pr_auc": average_precision_score(y, prob)}

baseline_prob = test["win_pct"].to_numpy()
logit_prob = logit.predict_proba(X_test)[:, 1]
forest_prob = forest.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    score_model("Baseline (best win percentage)", y_test,
                pick_one_per_season(test, baseline_prob), baseline_prob),
    score_model("Logistic regression", y_test,
                pick_one_per_season(test, logit_prob), logit_prob),
    score_model("Random forest", y_test,
                pick_one_per_season(test, forest_prob), forest_prob)
])
results.round(3)

                             model  accuracy  balanced_accuracy  precision  recall   f1  roc_auc  pr_auc\n0  Baseline (best win percentage)     0.903              0.674        0.4     0.4  0.4    0.663   0.344\n1            Logistic regression     0.871              0.565        0.2     0.2  0.2    0.691   0.204\n2                  Random forest     0.839              0.456        0.0     0.0  0.0    0.635   0.150

## 6. Model evaluation and selection

The simple baseline correctly selected 2 of the 5 champions in the holdout period. Logistic regression selected 1 of 5, while the random forest selected none. Logistic regression nevertheless produced the best ROC-AUC (0.691), meaning its overall ranking contained some useful signal even though its top choice was usually wrong.

I select **logistic regression** as the final trained model because it ranked winners better overall than the random forest and is easier to interpret. However, the baseline remains the strongest practical rule for choosing one champion in this small test period. This honest comparison is evidence that added complexity did not improve the decision.

In [ ]:
comparison = test[["yearID", "name", "world_series_winner", "win_pct"]].copy()
comparison["logit_probability"] = logit_prob
comparison["forest_probability"] = forest_prob
comparison.sort_values(["yearID", "logit_probability"], ascending=[True, False]).groupby("yearID").head(3)

## 7. Model interpretation

Positive logistic coefficients increase a team's predicted championship probability after standardization; negative coefficients decrease it while holding the other features constant. Coefficients should not be interpreted causally. Correlated predictors such as win percentage, OPS, and run differential can also make individual coefficient signs unstable.

In [ ]:
logit_importance = pd.DataFrame({
    "feature": FEATURES,
    "standardized_coefficient": logit.named_steps["model"].coef_[0],
    "random_forest_importance": forest.named_steps["model"].feature_importances_
}).sort_values("standardized_coefficient", key=abs, ascending=False)
logit_importance

In [ ]:
logit_pred = pick_one_per_season(test, logit_prob)
cm = confusion_matrix(y_test, logit_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False)
plt.xlabel("Predicted class"); plt.ylabel("Actual class")
plt.title("Logistic-regression confusion matrix");

## 8. Limitations, ethics, and reflection

- Twenty seasons contain only 20 positive examples, so estimates are unstable.
- The 2020 season had only 60 games and an expanded postseason. Rate statistics improve comparability but cannot remove the structural difference.
- MLB expanded the postseason format during this period, changing both the number and quality of playoff teams.
- Regular-season totals omit injuries, roster changes, starting-pitcher availability, opponent matchups, and short-series randomness.
- Team statistics are correlated. Feature importance describes model behavior, not causal effects.
- False predictions could mislead fans or bettors. This model should not be used for gambling, financial decisions, or personnel decisions.

The main result is not that machine learning can reliably forecast the champion. Instead, the analysis shows that regular-season quality carries some predictive signal, while postseason outcomes remain difficult to forecast. A future version should add preseason odds, roster health, playoff rotation strength, and season-aware cross-validation.

## 9. Sources and AI transparency

The statistical fields are derived from Sean Lahman's Baseball Database. The two repository snapshots used to assemble the 2005–2024 CSV are linked in the project README. MLB's glossary informed the OPS definition, and MLB's postseason-history documentation explains the format changes during the study period.

Generative AI disclosure: OpenAI ChatGPT/Codex (GPT-5 family, accessed September 2026) assisted with code structure, debugging, explanatory drafting, and website formatting. Nick Hawkins selected the research topic, reviewed the analysis, and is responsible for checking the results and explaining the work. AI was not used as a data source, and the numerical results were generated by the Python code in this repository.